# Henon Map and Rossler Attractor

## Overview

This notebook covers two additional chaotic systems used in QRC benchmarking:

- **Henon Map**: a 2D discrete-time chaotic map, extremely fast to generate, used in early RC papers
- **Rossler Attractor**: a continuous-time 3D system similar to Lorenz but with a simpler structure and slower chaos

Both appear in Fujii & Nakajima (2020) Table 1 and are used to test whether a reservoir generalises across different chaotic regimes.

---

# Part 1: Henon Map

## Background

Proposed by Michel Henon (1976) as a simplified model of the Poincare section of the Lorenz attractor. It is a 2D area-contracting map:

x(n+1) = 1 - a * x(n)^2 + y(n)
y(n+1) = b * x(n)

Standard chaotic parameters:

| Param | Value | Effect |
|-------|-------|-------|
| a     | 1.4   | controls folding (chaos strength) |
| b     | 0.3   | contraction (dissipation) |

With these values: **Lyapunov exponent ~0.42**, strange attractor with fractal dimension ~1.26.

### Advantages for QRC benchmarking
- **Discrete-time**: no ODE solver needed, just iterate the map
- **Very fast**: generate millions of points in milliseconds
- **Well-documented**: NRMSE benchmarks in most RC papers
- **Bivariate**: (x, y) pair -- can test cross-variable prediction

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.integrate import solve_ivp

np.random.seed(42)

In [ ]:
def henon_map(T=10000, a=1.4, b=0.3, x0=0.0, y0=0.0, washout=500):
    """
    Generate the Henon map trajectory.

    Parameters
    ----------
    T       : steps to return (after washout)
    a, b    : Henon parameters (a=1.4, b=0.3 gives chaos)
    x0, y0  : initial conditions
    washout : transient steps to discard

    Returns
    -------
    xy : np.ndarray, shape (T, 2)  -- columns: [x, y]
    """
    total = T + washout
    xy = np.zeros((total, 2))
    xy[0] = [x0, y0]

    for i in range(1, total):
        x_prev, y_prev = xy[i - 1]
        xy[i, 0] = 1.0 - a * x_prev ** 2 + y_prev
        xy[i, 1] = b * x_prev

    return xy[washout:]


henon = henon_map(T=10000)
print(f'Henon shape: {henon.shape}')
print(f'x: [{henon[:,0].min():.4f}, {henon[:,0].max():.4f}]')
print(f'y: [{henon[:,1].min():.4f}, {henon[:,1].max():.4f}]')

## Visualisation: Henon Attractor

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Henon Map', fontsize=13, fontweight='bold')

# Phase portrait
axes[0].scatter(henon[:, 0], henon[:, 1], s=0.3, alpha=0.4, color='steelblue')
axes[0].set_title('Phase portrait (Henon attractor)')
axes[0].set_xlabel('x(n)'); axes[0].set_ylabel('y(n)')
axes[0].grid(True, alpha=0.3)

# x time series
axes[1].plot(henon[:200, 0], lw=0.8, color='steelblue')
axes[1].set_title('x(n) time series')
axes[1].set_xlabel('n'); axes[1].set_ylabel('x(n)')
axes[1].grid(True, alpha=0.3)

# Return map: x(n) vs x(n+1)
axes[2].scatter(henon[:-1, 0], henon[1:, 0], s=0.5, alpha=0.4, color='darkorange')
axes[2].set_title('Return map: x(n) vs x(n+1)')
axes[2].set_xlabel('x(n)'); axes[2].set_ylabel('x(n+1)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('henon_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## Henon Train/Val/Test Split

In [ ]:
def nrmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2) / np.var(y_true)))


TRAIN, VAL, TEST = 5000, 1000, 2000

# Task: predict x(n+1) from x(n)
X_all = henon[:-1, 0]
y_all = henon[1:, 0]

X_train, y_train = X_all[:TRAIN], y_all[:TRAIN]
X_val,   y_val   = X_all[TRAIN:TRAIN+VAL], y_all[TRAIN:TRAIN+VAL]
X_test,  y_test  = X_all[TRAIN+VAL:TRAIN+VAL+TEST], y_all[TRAIN+VAL:TRAIN+VAL+TEST]

nrmse_naive = nrmse(y_test, X_test)
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print(f'Naive NRMSE: {nrmse_naive:.4f}')

---

# Part 2: Rossler Attractor

## Background

Proposed by Otto Rossler (1976) as a minimal system exhibiting chaos. It is simpler than Lorenz -- only one nonlinear term -- making it useful for studying the onset of chaos.

dx/dt = -(y + z)
dy/dt = x + a * y
dz/dt = b + z * (x - c)

Standard chaotic parameters:

| Param | Value | |
|-------|-------|-|
| a     | 0.2   | y-feedback |
| b     | 0.2   | z-source |
| c     | 5.7   | z-sink threshold |

### Differences from Lorenz
- **Slower chaos**: Lyapunov exponent ~0.07 (vs 0.9 for Lorenz) -- longer valid prediction horizon
- **Simpler geometry**: single spiral attractor (not double-wing)
- **One nonlinear term**: z*(x-c) -- lower complexity
- **Used in QRC**: Fujii & Nakajima (2020) Table 1 includes Rossler

Rossler is a good **intermediate test**: harder than periodic but easier than Lorenz.

In [ ]:
def rossler_rhs(t, state, a=0.2, b=0.2, c=5.7):
    """Right-hand side of the Rossler system."""
    x, y, z = state
    return [
        -(y + z),
        x + a * y,
        b + z * (x - c)
    ]


def generate_rossler(T_total=500.0, dt=0.05, x0=(0.0, 1.0, 0.0), a=0.2, b=0.2, c=5.7, washout_time=50.0):
    """
    Integrate the Rossler system.

    Parameters
    ----------
    T_total      : total integration time after washout
    dt           : output timestep
    x0           : initial condition (x, y, z)
    a, b, c      : Rossler parameters
    washout_time : transient time to discard

    Returns
    -------
    t_out : np.ndarray shape (N,)
    xyz   : np.ndarray shape (N, 3)
    """
    t_span = (0, T_total + washout_time)
    t_eval = np.arange(0, T_total + washout_time, dt)

    sol = solve_ivp(
        rossler_rhs, t_span, x0,
        method='RK45', t_eval=t_eval,
        args=(a, b, c), rtol=1e-9, atol=1e-9
    )

    washout_idx = int(washout_time / dt)
    xyz = sol.y[:, washout_idx:].T
    t_out = sol.t[washout_idx:] - washout_time
    return t_out, xyz


t_ros, xyz_ros = generate_rossler(T_total=500.0, dt=0.05)
x_ros, y_ros, z_ros = xyz_ros[:, 0], xyz_ros[:, 1], xyz_ros[:, 2]

print(f'Rossler trajectory: {xyz_ros.shape}')
print(f'Time span: {t_ros[0]:.1f} to {t_ros[-1]:.1f}')
print(f'x: [{x_ros.min():.2f}, {x_ros.max():.2f}]  '
      f'y: [{y_ros.min():.2f}, {y_ros.max():.2f}]  '
      f'z: [{z_ros.min():.2f}, {z_ros.max():.2f}]')

## Visualisation: Rossler Attractor

In [ ]:
fig = plt.figure(figsize=(16, 10))
fig.suptitle('Rossler Attractor', fontsize=14, fontweight='bold')

ax3d = fig.add_subplot(2, 3, (1, 4), projection='3d')
ax3d.plot(x_ros, y_ros, z_ros, lw=0.4, alpha=0.8, color='darkorange')
ax3d.set_xlabel('x'); ax3d.set_ylabel('y'); ax3d.set_zlabel('z')
ax3d.set_title('3D Rossler Attractor')

n_plot = 500
for i, (data, label, color) in enumerate([
    (x_ros, 'x(t)', 'steelblue'),
    (y_ros, 'y(t)', 'darkorange'),
    (z_ros, 'z(t)', 'green')
]):
    ax = fig.add_subplot(2, 3, i + 3 + 1)
    ax.plot(t_ros[:n_plot], data[:n_plot], lw=0.8, color=color)
    ax.set_title(label)
    ax.set_xlabel('t'); ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rossler_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## Comparison: Lorenz vs Rossler Lyapunov Exponents

The Lyapunov exponent lambda_max determines how fast nearby trajectories diverge:

| System  | lambda_max | Valid prediction horizon (approx) |
|---------|-----------|-----------------------------------|
| Henon   | ~0.42      | short (discrete map)              |
| Lorenz  | ~0.90      | ~1 time unit per Lyapunov time    |
| Rossler | ~0.07      | ~14 time units per Lyapunov time  |
| MG-17   | ~0.005     | very long (weakly chaotic)        |

This means **Rossler allows longer prediction horizons** than Lorenz -- useful for testing multi-step QRC.

## Save Both Datasets

In [ ]:
# Save Henon
pd.DataFrame({'x': henon[:, 0], 'y': henon[:, 1]}).to_csv('henon.csv', index_label='n')

for name, X, y in [('train', X_train, y_train), ('val', X_val, y_val), ('test', X_test, y_test)]:
    pd.DataFrame({'x_n': X, 'x_n1': y}).to_csv(f'henon_{name}.csv', index=False)

# Save Rossler
N_ros = len(xyz_ros)
n_train_r = int(0.70 * N_ros)
n_val_r   = int(0.10 * N_ros)

pd.DataFrame({'t': t_ros, 'x': x_ros, 'y': y_ros, 'z': z_ros}).to_csv('rossler.csv', index=False)

for name, start, end in [
    ('train', 0, n_train_r),
    ('val', n_train_r, n_train_r + n_val_r),
    ('test', n_train_r + n_val_r, N_ros)
]:
    pd.DataFrame({
        't': t_ros[start:end],
        'x': x_ros[start:end],
        'y': y_ros[start:end],
        'z': z_ros[start:end]
    }).to_csv(f'rossler_{name}.csv', index=False)

print('Saved: henon.csv, henon_train/val/test.csv')
print('Saved: rossler.csv, rossler_train/val/test.csv')

## Summary: Which Dataset to Use When

| Dataset     | Difficulty | Best for |
|-------------|-----------|----------|
| NARMA-10    | Medium    | Measuring memory capacity, direct QRC comparison |
| Mackey-Glass tau=17 | Medium | Standard RC benchmark, compare published NRMSE |
| Henon       | Medium-Hard | Fast prototyping, discrete-time reservoir |
| Mackey-Glass tau=30 | Hard | Stress-test reservoir memory |
| Rossler     | Medium    | Multi-step prediction, longer valid time |
| Lorenz      | Hard      | Autonomous forecasting, attractor reconstruction |

**Recommended workflow:**
1. Start with **NARMA-10** -- confirms your reservoir has sufficient memory
2. Move to **Mackey-Glass tau=17** -- compare NRMSE against Table 1 of Fujii & Nakajima 2020
3. Try **Lorenz** autonomous prediction -- the most visually compelling QRC demo
4. Use **Henon** for ablation studies (fast to run many experiments)

Key reference: Fujii & Nakajima (2020) Table 1 reports NRMSE for all four systems.